In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_csv('/media/prince/5A4E832F4E83034D/Movie recomender/zDeployment/Extra/main.csv')
embd = np.load('/media/prince/5A4E832F4E83034D/Movie recomender/zDeployment/Extra/final_vectors.npy', allow_pickle=True)

In [3]:
similarity_matrix = cosine_similarity(embd)
def recommend(movie, n=10):
    movie = movie.lower()
    index = df[df['title'].str.lower() == movie].index[0]
    score = similarity_matrix[index]
    similar_movies_index = np.argsort(score)[::-1][1:n+1]

    return df['title'].iloc[similar_movies_index].tolist()

In [4]:
recommend('thor')

['Iron Man',
 'The Avengers',
 'World War Z',
 'Iron Man 2',
 'Suicide Squad',
 'Inception',
 'Captain America: The First Avenger',
 'Prometheus',
 'Spider-Man',
 '300']

In [5]:
def recommend_by_genre_separate_clean(df, favorite_movies, top_n_genres=3, rec_per_genre=10):
    favorite_movies = [m.lower() for m in favorite_movies]

    fav_rows = df[df['title'].str.lower().isin(favorite_movies)]
    if fav_rows.empty:
        return []

    from collections import Counter
    genre_counter = Counter()

    # Extract genres from favorite movies
    for genres in fav_rows['genre_names']:
        if not genres or isinstance(genres, float):
            genres = []
        elif isinstance(genres, str):
            genres = [g.strip() for g in genres.split(',')]
        genre_counter.update(genres)

    # Top genres
    top_genres = [g for g, _ in genre_counter.most_common(top_n_genres)]

    # Dictionary to hold movies per genre
    genre_movies = {genre: [] for genre in top_genres}

    # Fill genre-based recommendation lists
    for idx, row in df.iterrows():
        title = row['title']
        genres = row['genre_names']

        if not genres or isinstance(genres, float):
            genres = []
        elif isinstance(genres, str):
            genres = [g.strip() for g in genres.split(',')]

        if title.lower() in favorite_movies:
            continue

        for g in top_genres:
            if g in genres and len(genre_movies[g]) < rec_per_genre:
                genre_movies[g].append(title)

    # 🔥 Convert to the clean `.tolist()` formatted output
    output_list = []
    for g in top_genres:
        movies = ", ".join(genre_movies[g])
        output_list.append(f"{g} = [{movies}]")

    return output_list


In [6]:
fav_movies = ['inside out', 'big here 6', 'frozen','Kung Fu Panda 2', 'Tangled']
recommend_by_genre_separate_clean(df, fav_movies)

['Animation = [Monsters University, Cars 2, Toy Story 3, The Good Dinosaur, Brave, WALL·E, A Christmas Carol, Up, Monsters vs Aliens, Shrek Forever After]',
 'Family = [Harry Potter and the Half-Blood Prince, The Chronicles of Narnia: Prince Caspian, Alice in Wonderland, Monsters University, Oz: The Great and Powerful, Cars 2, Toy Story 3, Jack the Giant Slayer, The Good Dinosaur, Brave]',
 'Drama = [The Dark Knight Rises, King Kong, Titanic, World War Z, The Great Gatsby, A Christmas Carol, The Dark Knight, Hugo, The Jungle Book, Snow White and the Huntsman]']

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   original_language  4803 non-null   object
 1   overview           4803 non-null   object
 2   popularity         4803 non-null   object
 3   runtime            4803 non-null   object
 4   tagline            4803 non-null   object
 5   title              4803 non-null   object
 6   vote_average       4803 non-null   object
 7   vote_count         4803 non-null   object
 8   genre_names        4775 non-null   object
 9   keywords_names     4391 non-null   object
 10  movie_text         4803 non-null   object
dtypes: object(11)
memory usage: 412.9+ KB


In [8]:
df['row_index'] = range(len(df))

In [9]:
df['row_index']

0          0
1          1
2          2
3          3
4          4
        ... 
4798    4798
4799    4799
4800    4800
4801    4801
4802    4802
Name: row_index, Length: 4803, dtype: int64

In [12]:
df.to_csv('final_list_with_index.csv', index=False)

In [13]:
df2  = pd.read_csv('/media/prince/5A4E832F4E83034D/Movie recomender/zDeployment/Extra/final_list.csv')

In [14]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   original_language  4803 non-null   object
 1   overview           4803 non-null   object
 2   runtime            4803 non-null   object
 3   tagline            4803 non-null   object
 4   title              4803 non-null   object
 5   genre_names        4803 non-null   object
 6   movie_text         4803 non-null   object
 7   id                 4803 non-null   int64 
 8   poster_url         4779 non-null   object
dtypes: int64(1), object(8)
memory usage: 337.8+ KB


In [15]:
df2['row_index'] = range(len(df))

In [16]:
df2['row_index']

0          0
1          1
2          2
3          3
4          4
        ... 
4798    4798
4799    4799
4800    4800
4801    4801
4802    4802
Name: row_index, Length: 4803, dtype: int64

In [18]:
df2.to_csv('final_list_with_index.csv', index=False)